# TF-IDF

Based on Term Frequency/Inverted Index last time

The text files are placed in folder './iitfidf', i.e. we have three files in the folder: 1.txt, 2.txt, 3.txt

The wholeTextFiles will read the files, and output a tuple (filename, fulltext)

data (i.e., wholeTextFiles) -> (filename, fulltext)

Here we calculate the tf "Term Frequency":
**tf = 1+log10(wc)**

In [1]:
import re
def removeSomeTexts(text):
    return re.sub(r"[^a-z-'\s]",'', text)

In [2]:
from pyspark import SparkConf, SparkContext
import os, re

from pyspark.sql.session import SparkSession

spark = SparkSession.builder.master('local').appName('MyTf-Idf').getOrCreate()
    
sc = spark.sparkContext
#sc = SparkContext.getOrCreate(SparkConf())
sc._conf.set('spark.executor.memory', '8g')
sc._conf.set('spark.driver.memory', '8g')
sc._conf.set('spark.driver.maxResultsSize', '0')

#data = sc.wholeTextFiles('data/iitfidf')
data = sc.wholeTextFiles('data/search')

numFiles = data.count()

# ((filename, token), TF)
wordcount = data.flatMap(lambda x: [((os.path.basename(x[0]), removeSomeTexts(i.lower())), 1) \
        for i in re.split('\W+', x[1])]) \
        .reduceByKey(lambda a, b: a + b)
    
tf1 = wordcount.map(lambda x: (x[0][1],(x[0][0],x[1]))).filter(lambda x: x[0] != '')

tf1.take(10)

[('title', ('1080-0.txt', 2)),
 ('a', ('1080-0.txt', 91)),
 ('modest', ('1080-0.txt', 1)),
 ('proposal', ('1080-0.txt', 5)),
 ('for', ('1080-0.txt', 48)),
 ('preventing', ('1080-0.txt', 2)),
 ('the', ('1080-0.txt', 171)),
 ('children', ('1080-0.txt', 20)),
 ('of', ('1080-0.txt', 138)),
 ('poor', ('1080-0.txt', 7))]

In [3]:
# tf = 1+log10(wc)
import math
tf = tf1.map(lambda x: (x[0],(x[1][0], 1+math.log10((x[1][1]) ))))

tf.take(10)

[('title', ('1080-0.txt', 1.3010299956639813)),
 ('a', ('1080-0.txt', 2.959041392321094)),
 ('modest', ('1080-0.txt', 1.0)),
 ('proposal', ('1080-0.txt', 1.6989700043360187)),
 ('for', ('1080-0.txt', 2.681241237375587)),
 ('preventing', ('1080-0.txt', 1.3010299956639813)),
 ('the', ('1080-0.txt', 3.2329961103921536)),
 ('children', ('1080-0.txt', 2.3010299956639813)),
 ('of', ('1080-0.txt', 3.1398790864012365)),
 ('poor', ('1080-0.txt', 1.845098040014257))]

# Computing Inverse Document Frequency (IDF)

The IDF score tells us how important is a particular word in the whole corpus. For instance, if a word is appearing in each document then the IDF score will be zero for that.

We know that the total number of documents for our case is 3. So, our next step will be to compute how many documents contain a particular token w. Further, we can use this information to calculate the IDF scores for all the tokens in the corpus.

We will be chaining the previous map/reduce computation to calculate IDF in this section. After the reduced step in the previous section, we obtained key/value pairs for respective tokens along with its term frequency.

In the first step, we will map the previous key/value pair to a new key/value pair. Here, the key will the token and its value will be the document id TF for that token along with a counter of 1. This one here indicates the presence of a word in document id associated with that.

((token, (filename, TF)) -> (token, 1)

Then in the second step, we will reduce by key to obtain the count of documents containing a particular token w (wordcount).

(token, 1) -> (token, 1+1+1+...)

Now in the third step, since we have the number of documents containing each token w, we will just map this final output with the logarithmic transformation to compute the IDF score.

(token, doc-freq-containing-token) -> (token, IDF)

In [4]:
idf2 = tf1.map(lambda x: (x[0], 1))
idf2.take(10)

[('title', 1),
 ('a', 1),
 ('modest', 1),
 ('proposal', 1),
 ('for', 1),
 ('preventing', 1),
 ('the', 1),
 ('children', 1),
 ('of', 1),
 ('poor', 1)]

In [5]:
idf3 = idf2.reduceByKey(lambda x, y: x + y)
idf3.take(10)

[('title', 15),
 ('a', 16),
 ('modest', 11),
 ('proposal', 11),
 ('for', 16),
 ('preventing', 4),
 ('the', 16),
 ('children', 16),
 ('of', 16),
 ('poor', 16)]

In [6]:
idf = idf3.map(lambda x: (x[0], math.log10(numFiles / (x[1]))))
idf.take(10)

[('title', 0.028028723600243534),
 ('a', 0.0),
 ('modest', 0.16272729749769976),
 ('proposal', 0.16272729749769976),
 ('for', 0.0),
 ('preventing', 0.6020599913279624),
 ('the', 0.0),
 ('children', 0.0),
 ('of', 0.0),
 ('poor', 0.0)]

## Computing TF-IDF

Now we have two RDDs one having the document id along with its document term frequency for each token. Another RDD having IDF scores for each token.

First, we will perform an inner join to assign each token with a document id, TF, and IDF score.

Second, we will map this function to multiply TF and IDF values of each token associated with respective document id.

Third, we sort the key

New Key/Value pair -> (filename, (token, TF-IDF))

In [7]:
tfidf = tf.join(idf)
tfidf.take(100)

[('title', (('1080-0.txt', 1.3010299956639813), 0.028028723600243534)),
 ('title', (('120-0.txt', 1.3010299956639813), 0.028028723600243534)),
 ('title', (('1342-0.txt', 1.3010299956639813), 0.028028723600243534)),
 ('title', (('1400-8.txt', 1.4771212547196624), 0.028028723600243534)),
 ('title', (('203-0.txt', 1.3010299956639813), 0.028028723600243534)),
 ('title', (('2600-0.txt', 2.0), 0.028028723600243534)),
 ('title', (('36-0.txt', 1.0), 0.028028723600243534)),
 ('title', (('74-0.txt', 1.0), 0.028028723600243534)),
 ('title', (('76-0.txt', 1.3010299956639813), 0.028028723600243534)),
 ('title', (('98-0.txt', 1.4771212547196624), 0.028028723600243534)),
 ('title', (('dolls_house.txt', 1.0), 0.028028723600243534)),
 ('title', (('frankenstein.txt', 1.3010299956639813), 0.028028723600243534)),
 ('title', (('monte_cristo.txt', 2.633468455579586), 0.028028723600243534)),
 ('title', (('sherlock.txt', 1.4771212547196624), 0.028028723600243534)),
 ('title', (('ulysses.txt', 1.90308998699194

In [8]:
tfidf = tfidf.map(lambda x: (x[1][0][0], (x[0], x[1][0][1] * x[1][1]))).sortByKey()
tfidf.take(50)

[('1080-0.txt', ('title', 0.03646621014409177)),
 ('1080-0.txt', ('modest', 0.16272729749769976)),
 ('1080-0.txt', ('preventing', 0.7832981079068755)),
 ('1080-0.txt', ('of', 0.0)),
 ('1080-0.txt', ('poor', 0.0)),
 ('1080-0.txt', ('in', 0.0)),
 ('1080-0.txt', ('ireland', 0.8893156098068689)),
 ('1080-0.txt', ('country', 0.0)),
 ('1080-0.txt', ('making', 0.0)),
 ('1080-0.txt', ('them', 0.0)),
 ('1080-0.txt', ('beneficial', 0.46709831647836336)),
 ('1080-0.txt', ('is', 0.0)),
 ('1080-0.txt', ('melancholy', 0.09017663034908803)),
 ('1080-0.txt', ('walk', 0.0)),
 ('1080-0.txt', ('this', 0.0)),
 ('1080-0.txt', ('when', 0.0)),
 ('1080-0.txt', ('streets', 0.11732250099206644)),
 ('1080-0.txt', ('roads', 0.12493873660829993)),
 ('1080-0.txt', ('female', 0.2041199826559248)),
 ('1080-0.txt', ('three', 0.0)),
 ('1080-0.txt', ('four', 0.0)),
 ('1080-0.txt', ('rags', 0.24036794985694204)),
 ('1080-0.txt', ('passenger', 0.2041199826559248)),
 ('1080-0.txt', ('an', 0.0)),
 ('1080-0.txt', ('these', 0

In [9]:
# create an inverted index 
index = tfidf.map(lambda x:(x[1][0].lower(), (x[0], x[1][1]))).filter(lambda x: x[0]!='')
index.take(50)

[('title', ('1080-0.txt', 0.03646621014409177)),
 ('modest', ('1080-0.txt', 0.16272729749769976)),
 ('preventing', ('1080-0.txt', 0.7832981079068755)),
 ('of', ('1080-0.txt', 0.0)),
 ('poor', ('1080-0.txt', 0.0)),
 ('in', ('1080-0.txt', 0.0)),
 ('ireland', ('1080-0.txt', 0.8893156098068689)),
 ('country', ('1080-0.txt', 0.0)),
 ('making', ('1080-0.txt', 0.0)),
 ('them', ('1080-0.txt', 0.0)),
 ('beneficial', ('1080-0.txt', 0.46709831647836336)),
 ('is', ('1080-0.txt', 0.0)),
 ('melancholy', ('1080-0.txt', 0.09017663034908803)),
 ('walk', ('1080-0.txt', 0.0)),
 ('this', ('1080-0.txt', 0.0)),
 ('when', ('1080-0.txt', 0.0)),
 ('streets', ('1080-0.txt', 0.11732250099206644)),
 ('roads', ('1080-0.txt', 0.12493873660829993)),
 ('female', ('1080-0.txt', 0.2041199826559248)),
 ('three', ('1080-0.txt', 0.0)),
 ('four', ('1080-0.txt', 0.0)),
 ('rags', ('1080-0.txt', 0.24036794985694204)),
 ('passenger', ('1080-0.txt', 0.2041199826559248)),
 ('an', ('1080-0.txt', 0.0)),
 ('these', ('1080-0.txt', 0

In [10]:
index = index.sortByKey()
index.take(100)

[('a', ('1080-0.txt', 0.0)),
 ('a', ('11-0.txt', 0.0)),
 ('a', ('120-0.txt', 0.0)),
 ('a', ('1342-0.txt', 0.0)),
 ('a', ('1400-8.txt', 0.0)),
 ('a', ('203-0.txt', 0.0)),
 ('a', ('2600-0.txt', 0.0)),
 ('a', ('36-0.txt', 0.0)),
 ('a', ('74-0.txt', 0.0)),
 ('a', ('76-0.txt', 0.0)),
 ('a', ('98-0.txt', 0.0)),
 ('a', ('dolls_house.txt', 0.0)),
 ('a', ('frankenstein.txt', 0.0)),
 ('a', ('monte_cristo.txt', 0.0)),
 ('a', ('sherlock.txt', 0.0)),
 ('a', ('ulysses.txt', 0.0)),
 ('aah', ('2600-0.txt', 1.2041199826559248)),
 ('aaron', ('monte_cristo.txt', 0.9030899869919435)),
 ('aaron', ('ulysses.txt', 1.1749471618603131)),
 ('ab', ('2600-0.txt', 0.9030899869919435)),
 ('ab', ('76-0.txt', 1.3339734147103033)),
 ('aback', ('120-0.txt', 0.7461677697975099)),
 ('aback', ('1400-8.txt', 0.505149978319906)),
 ('aback', ('2600-0.txt', 0.7461677697975099)),
 ('aback', ('98-0.txt', 0.505149978319906)),
 ('aback', ('ulysses.txt', 0.505149978319906)),
 ('abacus', ('2600-0.txt', 1.2041199826559248)),
 ('abaf

In [11]:
#save tfidf index to a text folder in your local directory
#lines = index.map(lambda x: (x[0], x[1][0], x[1][1]))
#lines.saveAsTextFile('./data/tfidf_txt2')

In [12]:
#save tfidf index as parquet format in your local directory
lines = index.map(lambda x: (x[0], x[1][0], x[1][1])).toDF()
lines.write.save("./data/tfidf_index2")